In [6]:
# 导入必要的库
import akshare as ak
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# print("akshare 版本:", ak.__version__)

## 步骤0: 获取可用的美股代码列表

**重要提示**: 根据 akshare 文档，股票代码应该通过 `stock_us_spot_em()` 函数获取。这个函数返回所有可用的美股代码列表。

In [7]:
# 获取所有美股的实时行情数据，从中可以获取股票代码
# 参考文档: https://akshare.akfamily.xyz/data/stock/stock.html#id2

import os
from datetime import datetime

# 定义保存路径
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
csv_file = os.path.join(data_dir, "stock_us_spot_em.csv")
pickle_file = os.path.join(data_dir, "stock_us_spot_em.pkl")

# 尝试从本地加载
stock_us_spot_df = None
if os.path.exists(csv_file):
    try:
        print(f"从本地加载数据: {csv_file}")
        stock_us_spot_df = pd.read_csv(csv_file)
        file_time = datetime.fromtimestamp(os.path.getmtime(csv_file))
        print(f"✓ 成功加载 {len(stock_us_spot_df)} 只股票代码 (保存时间: {file_time.strftime('%Y-%m-%d %H:%M:%S')})")
        print("提示: 如需更新数据，请删除文件后重新运行")
    except Exception as e:
        print(f"⚠ 加载本地数据失败: {e}，将从网络获取")

# 如果本地没有数据，从网络获取
if stock_us_spot_df is None:
    try:
        print("\n正在从网络获取美股代码列表...")
        
        # 修复 tqdm/ipywidgets 错误：Monkey patch tqdm.notebook 模块
        # 问题：akshare 内部使用 tqdm，在 notebook 环境中会尝试使用 ipywidgets
        # 解决：在调用前临时替换 tqdm.notebook.tqdm 为标准 tqdm
        
        try:
            import tqdm
            from tqdm import tqdm as tqdm_std
            import tqdm.notebook as tqdm_nb
            
            # 保存原始函数
            original_tqdm_nb = tqdm_nb.tqdm if hasattr(tqdm_nb, 'tqdm') else None
            original_status_printer = tqdm_nb.status_printer if hasattr(tqdm_nb, 'status_printer') else None
            
            # 创建一个安全的 status_printer，避免 ipywidgets 依赖
            def safe_status_printer(file, total, desc=None, ncols=None, **kwargs):
                # 返回一个简单的对象，模拟 tqdm 的容器
                class FakeContainer:
                    def __init__(self):
                        self.bar_style = ''
                    def __call__(self, *args, **kwargs):
                        return self
                    def update(self, *args, **kwargs):
                        pass
                    def close(self, *args, **kwargs):
                        pass
                return FakeContainer()
            
            # 替换 notebook.tqdm 为标准 tqdm
            tqdm_nb.tqdm = tqdm_std
            tqdm_nb.status_printer = safe_status_printer
            
            print("  ✓ 已修复 tqdm notebook 模式问题")
        except Exception as patch_error:
            print(f"  ⚠ tqdm patch 警告: {patch_error}")
            print("  提示: 如果仍然失败，请安装 ipywidgets: uv pip install ipywidgets")
        
        # 尝试获取数据
        stock_us_spot_df = ak.stock_us_spot_em()
        
        # 恢复原始设置（可选，通常不需要）
        # try:
        #     if original_tqdm_nb is not None:
        #         tqdm_nb.tqdm = original_tqdm_nb
        #     if original_status_printer is not None:
        #         tqdm_nb.status_printer = original_status_printer
        # except:
        #     pass
        
        print(f"✓ 成功获取 {len(stock_us_spot_df)} 只美股代码")
        
        # 保存到本地
        print(f"\n正在保存到本地...")
        stock_us_spot_df.to_csv(csv_file, index=False, encoding='utf-8-sig')
        stock_us_spot_df.to_pickle(pickle_file)
        print(f"✓ 已保存到:")
        print(f"  - CSV格式: {csv_file}")
        print(f"  - Pickle格式: {pickle_file}")
    except Exception as e:
        import traceback
        error_msg = str(e)
        print(f"\n❌ 获取美股代码列表失败: {error_msg}")
        
        # 检查是否是 tqdm/ipywidgets 错误
        if 'ipywidgets' in error_msg.lower() or 'iprogress' in error_msg.lower() or 'tqdm' in error_msg.lower():
            print("\n" + "="*60)
            print("🔧 解决方案（tqdm/ipywidgets 错误）:")
            print("="*60)
            print("方法1（推荐）: 安装 ipywidgets")
            print("  运行命令: uv pip install ipywidgets")
            print("  然后重新运行此 cell")
            print("\n方法2: 使用本地数据")
            print("  如果之前已经成功下载过数据，删除 data/stock_us_spot_em.csv")
            print("  然后重新运行，会从本地加载")
            print("\n方法3: 手动下载数据")
            print("  在 Python 脚本中运行（非 notebook 环境）:")
            print("  import akshare as ak")
            print("  df = ak.stock_us_spot_em()")
            print("  df.to_csv('data/stock_us_spot_em.csv', index=False, encoding='utf-8-sig')")
        else:
            print("\n详细错误信息:")
            traceback.print_exc()
            print("\n提示: 可以使用 stock_us_spot_em() 函数获取所有可用的美股代码")
        
        stock_us_spot_df = None

# 显示数据信息
if stock_us_spot_df is not None:
    print(f"\n{'='*60}")
    print(f"数据概览:")
    print(f"{'='*60}")
    print(f"列名: {list(stock_us_spot_df.columns)}")
    print("\n前10只股票:")
    display(stock_us_spot_df.head(10))
    
    # 查找一些知名公司的代码
    print("\n查找知名公司代码:")
    famous_companies = {
        '苹果': ['Apple', 'AAPL'],
        '微软': ['Microsoft', 'MSFT'],
        '谷歌': ['Google', 'GOOG', 'GOOGL'],
        '亚马逊': ['Amazon', 'AMZN'],
        '特斯拉': ['Tesla', 'TSLA'],
        'Meta': ['Meta', 'FB', 'META'],
        '英伟达': ['NVIDIA', 'NVDA'],
        '阿里巴巴': ['Alibaba', 'BABA']
    }
    
    print("\n" + "="*60)
    for company_name, keywords in famous_companies.items():
        matches = stock_us_spot_df[
            stock_us_spot_df['名称'].str.contains('|'.join(keywords), case=False, na=False) |
            stock_us_spot_df['代码'].str.contains('|'.join(keywords), case=False, na=False)
        ]
        if not matches.empty:
            print(f"\n{company_name}:")
            display(matches[['代码', '名称', '最新价']].head(3))
    
    # 使用说明
    print("\n" + "="*60)
    print("使用说明:")
    print("  1. 使用 stock_us_spot_df['代码'] 可以获取所有股票代码列表")
    print("  2. 代码示例: codes = stock_us_spot_df['代码'].tolist()")
    print("  3. 然后使用这些代码调用 stock_us_hist()")
    print(f"  4. 数据已保存到本地，下次可以直接加载")
    print(f"  5. 如需更新数据，删除 {csv_file} 后重新运行")
    
    # 创建 README 文件说明 data 目录
    readme_file = os.path.join(data_dir, "README.md")
    if not os.path.exists(readme_file):
        with open(readme_file, 'w', encoding='utf-8') as f:
            f.write("# 数据目录\n\n")
            f.write("此目录用于保存从 akshare 获取的美股代码列表数据。\n\n")
            f.write("## 文件说明\n\n")
            f.write("- `stock_us_spot_em.csv`: CSV 格式的美股代码列表（UTF-8编码）\n")
            f.write("- `stock_us_spot_em.pkl`: Pickle 格式的美股代码列表（保留数据类型）\n\n")
            f.write("## 使用说明\n\n")
            f.write("数据会在首次运行时自动下载并保存。\n")
            f.write("后续运行时会自动从本地加载，避免重复请求网络。\n")
            f.write("如需更新数据，删除对应的文件后重新运行代码即可。\n")
        print(f"\n✓ 已创建说明文件: {readme_file}")

从本地加载数据: data/stock_us_spot_em.csv
✓ 成功加载 12936 只股票代码 (保存时间: 2026-02-16 13:46:00)
提示: 如需更新数据，请删除文件后重新运行

数据概览:
列名: ['序号', '名称', '最新价', '涨跌额', '涨跌幅', '开盘价', '最高价', '最低价', '昨收价', '总市值', '市盈率', '成交量', '成交额', '振幅', '换手率', '代码']

前10只股票:


,序号,名称,最新价,涨跌额,涨跌幅,开盘价,最高价,最低价,昨收价,总市值,市盈率,成交量,成交额,振幅,换手率,代码
0,1,Algorhythm Holdings Inc,3.480,2.400,222.22,1.300,3.650,1.160,1.080,20038195.0,-0.70,167408697.0,438732384.0,230.56,2907.36,105.RIME
1,2,课标科技,3.810,2.120,125.44,2.120,4.060,1.800,1.690,89036599.0,-155.53,54029256.0,149472160.0,133.72,231.20,105.JDZG
2,3,Moolec Science SA Wt,0.019,0.010,118.39,0.025,0.028,0.012,0.009,NaN,NaN,5847663.0,124405.0,183.91,NaN,105.MLECW
3,4,Moolec Science SA,8.630,3.530,69.22,11.170,12.230,8.120,5.100,6266398.0,-0.06,50594663.0,498904512.0,80.59,6967.83,105.MLEC
4,5,Nuveen Real Asset Income and Gr,0.019,0.008,68.18,0.012,0.020,0.012,0.011,NaN,NaN,1211679.0,20088.0,71.82,NaN,106.JRIr
5,6,Atomera Inc,3.920,1.530,64.02,2.780,4.020,2.668,2.390,126827680.0,-6.22,28260518.0,101310694.0,56.56,87.35,105.ATOM
6,7,Fold Holdings Inc Wt,0.130,0.049,60.49,0.100,0.148,0.100,0.081,NaN,NaN,24065.0,2917.0,58.89,NaN,105.FLDDW
7,8,AIM ImmunoTech Inc,1.250,0.460,58.23,0.808,1.380,0.780,0.790,4001819.0,-0.25,3880740.0,4663604.0,75.94,121.22,107.AIM
8,9,Splash Beverage Group Inc,0.531,0.185,53.44,0.363,0.550,0.354,0.346,1543005.0,-0.05,2106767.0,1056311.0,56.65,72.49,107.SBEV
9,10,二倍做多RIVN ETF-GraniteShares,41.910,14.515,52.99,41.950,44.570,36.880,27.395,NaN,NaN,317219.0,12988728.0,28.07,NaN,105.RVNL



查找知名公司代码:


苹果:


,代码,名称,最新价
2737,107.PAPL,Pineapple Financial Inc,0.668
5995,106.APLE,Apple Hospitality REIT Inc,12.270
11155,105.AAPL,苹果,255.780



微软:


...


英伟达:


,代码,名称,最新价
10596,105.ANV,GraniteShares Autocallable NVDA,24.675
10911,105.LAYS,STKd 100% NVDA & 100% AMD ETF,41.821
10912,107.NVIT,YieldMax NVDA Performance & Dis,48.269



阿里巴巴:


,代码,名称,最新价
456,105.RYOJ,rYojbaba Co Ltd,2.78
10995,106.BABA,阿里巴巴,155.73
11239,107.BABW,Roundhill BABA WeeklyPay ETF,39.32



使用说明:
  1. 使用 stock_us_spot_df['代码'] 可以获取所有股票代码列表
  2. 代码示例: codes = stock_us_spot_df['代码'].tolist()
  3. 然后使用这些代码调用 stock_us_hist()
  4. 数据已保存到本地，下次可以直接加载
  5. 如需更新数据，删除 data/stock_us_spot_em.csv 后重新运行


In [8]:
# 方法1: 先从 stock_us_spot_em() 获取代码，然后使用该代码获取历史数据
# 这是推荐的方法，因为可以确保代码格式正确

try:
    # 步骤1: 获取股票代码列表（优先使用本地保存的数据）
    print("步骤1: 获取股票代码列表...")
    data_dir = "data"
    csv_file = os.path.join(data_dir, "stock_us_spot_em.csv")
    
    if os.path.exists(csv_file):
        print("  从本地加载数据...")
        stock_us_spot_df = pd.read_csv(csv_file)
        print(f"  ✓ 从本地加载 {len(stock_us_spot_df)} 只股票代码")
    else:
        print("  从网络获取数据...")
        stock_us_spot_df = ak.stock_us_spot_em()
        print(f"  ✓ 从网络获取 {len(stock_us_spot_df)} 只股票代码")
    
    # 步骤2: 查找苹果公司的代码
    print("\n步骤2: 查找苹果公司(AAPL)的代码...")
    apple_matches = stock_us_spot_df[
        stock_us_spot_df['名称'].str.contains('Apple', case=False, na=False) |
        stock_us_spot_df['代码'].str.contains('AAPL', case=False, na=False)
    ]
    
    if apple_matches.empty:
        print("未找到苹果公司，尝试使用常见代码格式...")
        # 尝试常见的代码格式
        test_symbols = ["AAPL", "105.AAPL"]
    else:
        # 使用 stock_us_spot_em() 返回的代码
        apple_code = apple_matches.iloc[0]['代码']
        print(f"找到苹果公司代码: {apple_code}")
        test_symbols = [apple_code, "AAPL", "105.AAPL"]  # 也尝试其他格式
    
    # 步骤3: 使用获取到的代码来获取历史数据
    print("\n步骤3: 获取历史数据...")
    stock_us_hist_df = None
    successful_symbol = None
    
    for symbol in test_symbols:
        try:
            print(f"  尝试代码: {symbol}")
            stock_us_hist_df = ak.stock_us_hist(
                symbol=symbol,
                period="daily",
                start_date="20240101",
                end_date="20241231",
                adjust=""  # 先尝试不复权
            )
            
            if stock_us_hist_df is not None and not stock_us_hist_df.empty:
                successful_symbol = symbol
                print(f"  ✓ 成功！")
                break
            elif stock_us_hist_df is not None and stock_us_hist_df.empty:
                print(f"  ⚠ 返回空数据")
            else:
                print(f"  ✗ 返回 None")
        except Exception as e:
            print(f"  ✗ 错误: {str(e)[:80]}")
            continue
    
    # 步骤4: 显示结果
    if stock_us_hist_df is not None and not stock_us_hist_df.empty:
        print(f"\n{'='*60}")
        print(f"✓ 成功获取数据 (使用代码: {successful_symbol})")
        print(f"{'='*60}")
        print(f"数据形状: {stock_us_hist_df.shape}")
        print(f"\n列名: {list(stock_us_hist_df.columns)}")
        print("\n前10条数据:")
        display(stock_us_hist_df.head(10))
        print("\n数据基本信息:")
        print(stock_us_hist_df.info())
    else:
        print(f"\n{'='*60}")
        print("❌ 所有代码格式都失败了")
        print(f"{'='*60}")
        print("建议：")
        print("1. 检查网络连接")
        print("2. 确认 stock_us_spot_em() 返回的代码格式")
        print("3. 尝试使用更近的日期范围")
        print("4. 更新 akshare: uv pip install akshare --upgrade")
    
except Exception as e:
    import traceback
    print(f"获取数据时出错: {e}")
    print("\n详细错误信息:")
    traceback.print_exc()

步骤1: 获取股票代码列表...
  从本地加载数据...
  ✓ 从本地加载 12936 只股票代码

步骤2: 查找苹果公司(AAPL)的代码...
找到苹果公司代码: 107.PAPL

步骤3: 获取历史数据...
  尝试代码: 107.PAPL
  ✓ 成功！

✓ 成功获取数据 (使用代码: 107.PAPL)
数据形状: (252, 11)

列名: ['日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']

前10条数据:


,日期,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,2024-01-02,1.700,1.79,1.800,1.620,27190,46169.0,10.06,0.00,0.00,0.38
1,2024-01-03,1.682,1.74,1.750,1.650,17652,29786.0,5.59,-2.79,-0.05,0.25
2,2024-01-04,1.680,1.67,1.750,1.625,33177,56116.0,7.18,-4.02,-0.07,0.46
3,2024-01-05,1.680,1.65,1.740,1.620,21718,36469.0,7.19,-1.20,-0.02,0.30
4,2024-01-08,1.650,1.63,1.740,1.603,15326,25308.0,8.30,-1.21,-0.02,0.21
5,2024-01-09,1.700,1.60,1.740,1.600,97455,164419.0,8.59,-1.84,-0.03,1.36
6,2024-01-10,1.610,1.62,1.670,1.610,22855,37262.0,3.75,1.25,0.02,0.32
7,2024-01-11,1.640,1.56,1.650,1.560,37918,60546.0,5.56,-3.70,-0.06,0.53
8,2024-01-12,1.600,1.53,1.615,1.511,17164,26476.0,6.67,-1.92,-0.03,0.24
9,2024-01-16,1.610,1.67,1.730,1.570,31175,52066.0,10.46,9.15,0.14,0.43



数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   日期      252 non-null    object 
 1   开盘      252 non-null    float64
 2   收盘      252 non-null    float64
 3   最高      252 non-null    float64
 4   最低      252 non-null    float64
 5   成交量     252 non-null    int64  
 6   成交额     252 non-null    float64
 7   振幅      252 non-null    float64
 8   涨跌幅     252 non-null    float64
 9   涨跌额     252 non-null    float64
 10  换手率     252 non-null    float64
dtypes: float64(9), int64(1), object(1)
memory usage: 21.8+ KB
None
